# 03 — Agent memory patterns: sessions, events, scratch

memtomem ships with first-class primitives for agent-style memory:

- **Sessions** record the lifespan of an agent run. Each session can hold a
  sequence of events.
- **Events** are timestamped records of what the agent did (`search`,
  `decide`, `observe`, …).
- **Scratch (working memory)** is a key-value store that lives for the
  duration of a task, optionally scoped to a session and with a TTL.

In this notebook we walk through one agent-style turn end-to-end, using the
storage mixins directly.

## Prerequisites

- **memtomem** installed:
  ```bash
  # From PyPI
  uv pip install "memtomem[ollama]" jupyter ipykernel
  # Or from a source checkout
  uv pip install -e "packages/memtomem[all]" jupyter ipykernel
  ```
- Ollama running with `nomic-embed-text` — same as notebooks 01 and 02.

In [1]:
# Verify Ollama is reachable before doing anything else.
import httpx

try:
    r = httpx.get("http://localhost:11434/api/tags", timeout=2)
    r.raise_for_status()
except Exception as e:
    raise RuntimeError(
        "Ollama is not reachable at http://localhost:11434.\n"
        "Start it and pull the default embedding model:\n"
        "  ollama serve\n"
        "  ollama pull nomic-embed-text\n"
        "then re-run this cell."
    ) from e

print("Ollama is up.")

Ollama is up.


## Step 1 — Set up components

In [2]:
import json
import os
import tempfile
from pathlib import Path

from memtomem.config import Mem2MemConfig
from memtomem.server.component_factory import (
    Components,
    create_components,
    close_components,
)

# 1. Create an isolated temp directory so nothing lands in ~/.memtomem.
tmp = tempfile.TemporaryDirectory()
tmp_path = Path(tmp.name)
db_path = tmp_path / "memory.db"
notes_dir = tmp_path / "notes"
notes_dir.mkdir()

# 2. Override config via environment variables. Note the double underscore —
#    pydantic-settings uses "__" as the nesting separator for section fields.
os.environ["MEMTOMEM_STORAGE__SQLITE_PATH"] = str(db_path)
os.environ["MEMTOMEM_INDEXING__MEMORY_DIRS"] = json.dumps([str(notes_dir)])
os.environ["MEMTOMEM_EMBEDDING__PROVIDER"] = "ollama"
os.environ["MEMTOMEM_EMBEDDING__MODEL"] = "nomic-embed-text"
os.environ["MEMTOMEM_EMBEDDING__DIMENSION"] = "768"

# 3. Apply the same fields directly on the config object, and temporarily
#    disable load_config_overrides() so the real ~/.memtomem/config.json
#    cannot leak into this notebook. This mirrors the pattern used in
#    packages/memtomem/tests/conftest.py.
config = Mem2MemConfig()
config.storage.sqlite_path = db_path
config.indexing.memory_dirs = [notes_dir]

import memtomem.config as _cfg

_orig_loader = _cfg.load_config_overrides
_cfg.load_config_overrides = lambda c: None
try:
    comp: Components = await create_components(config)
finally:
    _cfg.load_config_overrides = _orig_loader

print(f"memtomem components ready. db={db_path}")

memtomem components ready. db=/var/folders/4n/mt3km5rs1bn_b0w_d6hvlmhc0000gn/T/tmp_uif80e5/memory.db


## Step 2 — Start a session

Sessions need a UUID you supply yourself. The `agent_id` is a free-form
label for whichever agent is running, and `namespace` is a memtomem
namespace that lets you segment episodic memory by project.

In [3]:
import uuid
from datetime import datetime, timezone

session_id = str(uuid.uuid4())

await comp.storage.create_session(
    session_id=session_id,
    agent_id="demo-agent",
    namespace="research",
    metadata={"task": "investigate retrieval quality"},
)

print(f"session started: {session_id}")

session started: 757f0ca8-07f5-413b-8ebc-3e2b46e16cd1


## Step 3 — Log events as the agent works

Each call to `add_session_event()` appends a row to the session log. The
`event_type` is free-form; common values are `search`, `observe`, `decide`,
`tool_call`, `reflect`, `error`. The `chunk_ids` list lets you link an event
back to the retrieval result that justified it.

In [4]:
await comp.storage.add_session_event(
    session_id=session_id,
    event_type="search",
    content="hybrid retrieval tuning",
)

await comp.storage.add_session_event(
    session_id=session_id,
    event_type="observe",
    content="default rrf_k=60 works for short queries, suffers on long ones",
)

await comp.storage.add_session_event(
    session_id=session_id,
    event_type="decide",
    content="try rrf_k=30 + rerank=True for queries > 16 tokens",
)

print("three events logged")

three events logged


## Step 4 — Use scratch (working memory) for intermediate state

Scratch entries are meant for short-lived, task-scoped values — things you
would normally put on an agent's blackboard or in a dict passed through
node state. Scoping an entry to `session_id` lets memtomem clean it up when
the session ends.

In [5]:
await comp.storage.scratch_set(
    key="current_hypothesis",
    value="long queries benefit from lower rrf_k",
    session_id=session_id,
)

await comp.storage.scratch_set(
    key="pending_todo",
    value="re-run tuning on the 200-query eval set",
    session_id=session_id,
)

fetched = await comp.storage.scratch_get("current_hypothesis")
print("current_hypothesis:", fetched["value"] if fetched else "(missing)")

scratch_rows = await comp.storage.scratch_list(session_id=session_id)
print(f"scratch entries for this session: {len(scratch_rows)}")
for row in scratch_rows:
    print(f"  {row['key']:20s} -> {row['value']}")

current_hypothesis: long queries benefit from lower rrf_k
scratch entries for this session: 2
  current_hypothesis   -> long queries benefit from lower rrf_k
  pending_todo         -> re-run tuning on the 200-query eval set


## Step 5 — Replay the event log

At any time you can ask memtomem for the full event stream of a session.
This is how you would rebuild an agent's trajectory or show a human
reviewer what happened.

In [6]:
events = await comp.storage.get_session_events(session_id)

print(f"{len(events)} event(s) in session {session_id[:8]}…")
for e in events:
    print(f"  [{e['event_type']:8s}] {e['content']}")

3 event(s) in session 757f0ca8…
  [search  ] hybrid retrieval tuning
  [observe ] default rrf_k=60 works for short queries, suffers on long ones
  [decide  ] try rrf_k=30 + rerank=True for queries > 16 tokens


## Step 6 — End the session

`end_session()` records a summary plus arbitrary metadata (we tuck the
event type counts in there for bookkeeping). After the session ends the
scratch entries that were scoped to it can be dropped via
`scratch_cleanup(session_id=...)`.

In [7]:
event_counts: dict[str, int] = {}
for e in events:
    event_counts[e["event_type"]] = event_counts.get(e["event_type"], 0) + 1

await comp.storage.end_session(
    session_id=session_id,
    summary="explored rrf_k tradeoff, planned eval re-run",
    metadata={"event_counts": event_counts},
)

dropped = await comp.storage.scratch_cleanup(session_id=session_id)
print(f"session closed, {dropped} scratch entries cleaned up")
print("event counts:", event_counts)

session closed, 2 scratch entries cleaned up
event counts: {'search': 1, 'observe': 1, 'decide': 1}


## Step 7 — Time-based recall

`storage.recall_chunks()` is the building block behind the `mem_recall`
MCP tool. It returns chunks created inside a time window, with optional
source and namespace filters — no query embedding required. We will index
one note, then recall everything that was added in the last minute.

In [8]:
# Drop a fresh note so we have something to recall.
recall_note = notes_dir / "finding.md"
recall_note.write_text(
    "# Retrieval finding\n\n"
    "Hybrid mode beats dense-only on short factual queries but loses on "
    "long narrative queries, probably because BM25 saturation dominates.\n",
    encoding="utf-8",
)
await comp.index_engine.index_file(recall_note)

from datetime import timedelta

since = datetime.now(timezone.utc) - timedelta(minutes=1)

recent = await comp.storage.recall_chunks(since=since, limit=10)
print(f"{len(recent)} chunk(s) in the last minute:")
for chunk in recent:
    name = Path(chunk.metadata.source_file).name
    head = " / ".join(chunk.metadata.heading_hierarchy) or "(no heading)"
    print(f"  {name:20s} {head}")

[04/11/26 11:06:19] INFO     HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"  ]8;id=825719;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py\_client.py]8;;\:]8;id=290695;file:///Users/pdstudio/.cache/uv/archive-v0/O4Y8eIGPPvWNha15A38Ys/lib/python3.13/site-packages/httpx/_client.py#1740\1740]8;;\

1 chunk(s) in the last minute:
  finding.md           # Retrieval finding


## Mapping to the MCP tool surface

Every call above has an MCP-tool equivalent. In `core` tool mode these live
behind `mem_do`:

| Python call | MCP equivalent |
|-------------|----------------|
| `storage.create_session(...)` | `mem_do(action="session_start", params={...})` |
| `storage.add_session_event(...)` | `mem_do(action="session_log", params={...})` |
| `storage.get_session_events(...)` | `mem_do(action="session_events", params={...})` |
| `storage.end_session(...)` | `mem_do(action="session_end", params={...})` |
| `storage.scratch_set(...)` | `mem_do(action="scratch_set", params={...})` |
| `storage.scratch_get(...)` | `mem_do(action="scratch_get", params={...})` |
| `storage.recall_chunks(...)` | `mem_recall(...)` |

## Cleanup

In [9]:
# Release connections and remove the temp directory.
await close_components(comp)
tmp.cleanup()

# Clean up the environment variables we set.
for key in (
    "MEMTOMEM_STORAGE__SQLITE_PATH",
    "MEMTOMEM_INDEXING__MEMORY_DIRS",
    "MEMTOMEM_EMBEDDING__PROVIDER",
    "MEMTOMEM_EMBEDDING__MODEL",
    "MEMTOMEM_EMBEDDING__DIMENSION",
):
    os.environ.pop(key, None)

print("clean.")

clean.
